In [ ]:
# Final annotation and composition analysis of the heart single-cell atlas.
import os, glob
import scanpy as sc
import anndata as ad
import pandas as pd

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
input_dir = "/home/yanjun3/atlas/heart_sc/figures_annotation_sc"

In [ ]:
import seaborn as sns
import torch

In [ ]:
sc.set_figure_params(figsize=(3, 3),  facecolor="white",
    frameon=True,   dpi = 300, format = 'png')
sc.settings.figdir = "/home/yanjun3/atlas/heart_sc/figures_annotation_sc"
sns.set_theme()
torch.set_float32_matmul_precision("high")

%config InlineBackend.print_figure_kwargs={"facecolor": "w"}
%config InlineBackend.figure_format="retina"

In [ ]:
!pwd

In [ ]:
adata = sc.read('/Users/yanjun/Desktop/PQE_thesis/scripts/chapter5/qc3_scvi.h5ad')

# sc0.4 ; sn0.5

In [ ]:
adata

In [ ]:
cluster_map = {
    '0': "Mac0",
    '1': "Mac1",
    '2': "Mac2",
    '3': "Mac3",
    '4': "Mac4",
    '5': "Mac5",
    '7': "Mac6",
    '8': "Mac2"
}

adata.obs["anno_layer1"] = adata.obs["leiden_res0_4"].map(cluster_map)

In [ ]:
adata.obs

In [ ]:
sc.pl.umap(adata, 
          color = ['anno_layer1','Disease', 
                   ], size = 30,
           color_map = 'RdBu_r', ncols = 1,save = 'sc_umap2.png'
          )

In [ ]:
sc.pl.dotplot(adata, ["TNF", "IL1B", "CXCL8", "NFKBIA", "NLRP3", "CCL2"], groupby="anno_layer1")

In [ ]:
sc.pl.umap(adata, 
          color = ['anno_layer1', 
                  'LYVE1', 'C1QC', 'MRC1','F13A1', 'NRP1',   'CD68',
                   'FOLR2',  'G2M_score', 'anatomical_site', 'Age_1'], size = 30,
           color_map = 'RdBu_r', ncols = 2,save = 'sc_umap.png'
          )

In [ ]:
sc.pl.umap(adata, 
          color = [
                  'LYVE1', 'C1QA', 'NRP1', 'leiden_res0_4',  'CD68',
                  'C1QC', 'MRC1', 'SELENOP', 'F13A1', 'FOLR2', 'STIL', 'anno_layer1', 'anatomical_site', 'G2M_score'], size = 30,
           color_map = 'RdBu_r', ncols = 2
          )

In [ ]:
lyve1_genes = ["MRC1", "F13A1", "LYVE1", "FOLR2",]

sc.tl.score_genes(
    adata,
    gene_list=lyve1_genes,
    score_name="lyve1_core"
)

In [ ]:
sc.pl.matrixplot(
    adata,
    var_names=["lyve1_core", "LYVE1", "MRC1", "F13A1", "FOLR2"],
    groupby="anno_layer1",
    standard_scale="var",
    cmap = 'RdBu_r'
)

In [ ]:
adata

In [ ]:
sc.tl.rank_genes_groups(adata, groupby="anno_layer1", method="wilcoxon")
sc.pl.rank_genes_groups_dotplot(adata, n_genes=20, standard_scale='var', 
                                save = 'anno_layer1'
                               )

In [ ]:
result = adata.uns['rank_genes_groups']
groups = result['names'].dtype.names
df = pd.DataFrame(
    {f"{group}_{key}": result[key][group]
     for group in groups for key in ['names', 'scores', 'pvals', 'pvals_adj', 'logfoldchanges']}
)
df.to_csv(f"{input_dir}/anno_layer1.csv", index=False)

In [ ]:
genes = [
    "LYVE1", "C1QA", "NRP1", "CD68",
    "C1QC", "MRC1", "SELENOP", "F13A1", "FOLR2", "STIL"
]

sc.pl.matrixplot(
    adata,
    var_names=genes,
    groupby="anno_layer1",
    standard_scale="var",
    cmap="RdBu_r"
)

In [ ]:
markers_dict = {
    "LYVE1_signatures" : [
        'LYVE1', 'MRC1', 'F13A1', 'SELENOP', 'FOLR2', 'NRP1', 'CD68', 'C1QA'
    ],
    "oxphos_phosphorylation": [
        "COX4I1", "COX5B", "COX6B1", "COX7A2", "ATP5F1B",
        "ATP5F1C", "ATP5PO", "UQCRB", "NDUFA1", "SDHD"
    ],
    "antigen_presenting": [
        "CD74", "HLA-DRA", "HLA-DPA1", "HLA-DPB1", "HLA-DMA",
        "HLA-DMB", "HLA-DRB1", "FCER1G", "CTSS", "CD209"
    ],
    "actin_filament_organization": [
        "TMSB10", "TMSB4X", "AIF1", "TAGLN2", "PFN1",
        "CFL1", "ARPC1B", "ARPC3", "CDC42", "RAC1"
    ],
    "regulation_of_endocytosis": [
        "MERTK", "PICALM", "DOCK2", "RAB31", "AAK1",
        "SYK", "NUMB", "TLR2", "RIN3", "PLCG2"
    ],
    "clathrin_dependent_endocytosis": [
        "PICALM", "FCHO2", "ITSN1", "ITSN2", "AP2A2",
        "AAK1", "BMP2K", "FCHSD2", "SIGLEC1", "PIK3CB"
    ],
    "inflammatory_response": [
        "IL1B", "TNF", "CXCL8", "NFKBIA", "TNFAIP3",
        "NLRP3", "CCL2", "PTGS2", "TLR2", "CEBPB"
    ],
    "g2m_genes": [
        "MKI67", "TOP2A", "CENPF", "UBE2C", "BIRC5",
        "AURKB", "AURKA", "CDK1", "CDC20", "TPX2"
    ],
    "dna_replication": [
        "PCNA", "MCM2", "MCM5", "MCM6", "MCM7",
        "CDC6", "CDT1", "GINS1", "RRM2", "PCLAF"
    ],
    "detoxification":["HBA1", "HBA2", "HBB", "HBD", "HBG1", "HBG2",
    "HBM", "HBQ1", "HBZ"
        
    ],
    "regulation_of_apoptotic_signaling_pathway": [
        "LGALS3", "TREM2", "CTSS", "CTSH", "BAX",
        "IFI6", "TNFSF12", "FAIM", "HSPB1", "PARK7"
    ],
    "tumor_necrosis_factor_production": [
        "TREM2", "TYROBP", "GPNMB", "LPL", "CYBA",
        "LILRB4", "CD36", "SIRPA", "PTPN6", "LGALS9"
    ]
}

In [ ]:
order = ["Mac0", "Mac1", "Mac2", "Mac3", "Mac4", "Mac5", "Mac6"]

adata.obs["anno_layer1"] = adata.obs["anno_layer1"].astype("category")
adata.obs["anno_layer1"] = adata.obs["anno_layer1"].cat.reorder_categories(order, ordered=True)


In [ ]:
rank_markers_dict = {
    g: sc.get.rank_genes_groups_df(adata, group=g)["names"].head(10).tolist()
    for g in order
}

In [ ]:
mp = sc.pl.matrixplot(
    adata,
    var_names=rank_markers_dict,
    groupby="anno_layer1",
    standard_scale="var",
    cmap="RdBu_r",
    swap_axes=True,
    figsize=(7, 4),
    return_fig=True
)

mp.add_totals().style(edge_color="black")
mp.make_figure()

axes = mp.get_axes()

# gene names / tick labels
axes["mainplot_ax"].tick_params(axis="y", labelsize=4)
axes["mainplot_ax"].tick_params(axis="x", labelsize=8)

# module names
for ax_name, ax in axes.items():
    if ax_name != "group_extra_ax":
        for txt in ax.texts:
            txt.set_visible(False)
        ax.tick_params(
            axis="both",
            which="both",
            labelbottom=True,
            labeltop=False,
            labelleft=False,
            labelright=False)

In [ ]:
mp = sc.pl.matrixplot(
    adata,
    var_names=rank_markers_dict,
    groupby="anno_layer1",
    standard_scale="var",
    cmap="RdBu_r",
    swap_axes=True,
    figsize=(7, 6),
    return_fig=True
)

mp.add_totals().style(edge_color="black")
mp.make_figure()

axes = mp.get_axes()

axes["mainplot_ax"].tick_params(axis="y", labelsize=4)
axes["mainplot_ax"].tick_params(axis="x", labelsize=8)

for ax in axes.values():
    for txt in ax.texts:
        txt.set_fontsize(4)
        txt.set_rotation(0)
        txt.set_ha("center")
        txt.set_va("center")

plt.tight_layout()
plt.savefig("layer1_markers.pdf", bbox_inches="tight")
plt.show()

In [ ]:
mp = sc.pl.matrixplot(
    adata,
    var_names=markers_dict,
    groupby="anno_layer1",
    standard_scale="var",
    cmap="RdBu_r",
    swap_axes=True,
    figsize=(7, 8),
    return_fig=True
)

mp.add_totals().style(edge_color="black")
mp.make_figure()

axes = mp.get_axes()

axes["mainplot_ax"].tick_params(axis="y", labelsize=4)
axes["mainplot_ax"].tick_params(axis="x", labelsize=8)

for ax in axes.values():
    for txt in ax.texts:
        txt.set_fontsize(4)
        txt.set_rotation(0)
        txt.set_ha("center")
        txt.set_va("center")
cax = axes["color_legend_ax"]
pos = cax.get_position()
cax.set_position([pos.x0 + 0.02, pos.y0, pos.width * 0.7, pos.height * 0.6])


plt.tight_layout()
plt.savefig("layer1_function1.png", bbox_inches="tight", dpi = 600)
plt.show()

In [ ]:
mp = sc.pl.matrixplot(
    adata,
    var_names=markers_dict,
    groupby="anno_layer1",
    standard_scale="var",
    cmap="RdBu_r",
    swap_axes=True,
    figsize=(7, 4),
    return_fig=True
)

mp.add_totals().style(edge_color="black")
mp.make_figure()

axes = mp.get_axes()

# gene names / tick labels
axes["mainplot_ax"].tick_params(axis="y", labelsize=4)
axes["mainplot_ax"].tick_params(axis="x", labelsize=8)

# module names
for ax_name, ax in axes.items():
    if ax_name != "group_extra_ax":
        for txt in ax.texts:
            txt.set_visible(False)
        ax.tick_params(
            axis="both",
            which="both",
            labelbottom=True,
            labeltop=False,
            labelleft=False,
            labelright=False
        )
#mp.savefig("figures/layer1_function.png", dpi=300, bbox_inches="tight")
#mp.show()

In [ ]:
sc.pl.matrixplot(
    adata,
    var_names=markers_dict,
    groupby="anno_layer1",
    standard_scale="var",
    cmap="RdBu_r",
    save = 'layer1_function.png'
)

In [ ]:
info = pd.read_csv("/home/yanjun3/atlas/heart_sc/heart_sc_clean_.csv").drop_duplicates(subset=["sample_id"])

meta = info.set_index("sample_id")
new_cols = meta.columns.difference(adata.obs.columns)

adata.obs = adata.obs.join(meta[new_cols], on="sample")

In [ ]:
adata_healthy = adata[adata.obs["Disease"] == "healthy"].copy()

In [ ]:
adata_healthy

In [ ]:
adata_healthy.obs['Age_1'].unique()

In [ ]:
adata_pb = adata_healthy[adata_healthy.obs["Age_1"].isin(["adult", "feotus"])].copy()

In [ ]:
order = ["Mac0", "Mac1", "Mac2", "Mac3", "Mac4", "Mac5", "Mac6"]

adata_pb.obs["anno_layer1"] = pd.Categorical(
    adata_pb.obs["anno_layer1"],
    categories=order,
    ordered=True
)


import matplotlib.pyplot as plt
obs_df = adata_pb.obs[['Age_1', 'anno_layer1']]
frequency_df = obs_df.groupby(['Age_1', 'anno_layer1']).size().unstack(fill_value=0)
percentage_df = frequency_df.div(frequency_df.sum(axis=1), axis=0) * 100
percentage_df = percentage_df.iloc[::-1]

fig = plt.figure(figsize=(10, 3))
ax = percentage_df.plot(kind='barh', stacked=True, ax=fig.gca())  

plt.title("Proportion of clusters across age / clusters")
plt.xlabel("Percentage (%)")
plt.ylabel("sample")
plt.legend(title='Clusters', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(False)  
plt.xticks(rotation=0)  

plt.tight_layout()
plt.tight_layout()
plt.savefig("samplefreq.png", dpi=300)
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

# =========================
# 1. subset adult / feotus
# =========================
adata_pb = adata_healthy[adata_healthy.obs["Age_1"].isin(["adult", "feotus"])].copy()

celltypes_use = ["Mac0", "Mac1", "Mac3", "Mac4"]

obs = adata_pb.obs[["sample", "Age_1", "anno_layer1"]].copy()

# make sure each sample has one unique age label
sample_meta = obs[["sample", "Age_1"]].drop_duplicates()

# =========================
# 2. total cells per sample
#    denominator = all cells in that sample
# =========================
total_df = (
    obs.groupby(["sample", "Age_1"])
    .size()
    .reset_index(name="total_cells")
)

# =========================
# 3. count selected cell types per sample
# =========================
count_df = (
    obs[obs["anno_layer1"].isin(celltypes_use)]
    .groupby(["sample", "Age_1", "anno_layer1"])
    .size()
    .reset_index(name="n_cells")
)

# =========================
# 4. make sure missing sample-celltype combos are 0
# =========================
all_combos = (
    sample_meta.assign(key=1)
    .merge(pd.DataFrame({"anno_layer1": celltypes_use, "key": 1}), on="key")
    .drop(columns="key")
)

prop_df = (
    all_combos
    .merge(count_df, on=["sample", "Age_1", "anno_layer1"], how="left")
    .merge(total_df, on=["sample", "Age_1"], how="left")
)

prop_df["n_cells"] = prop_df["n_cells"].fillna(0)
prop_df["proportion"] = prop_df["n_cells"] / prop_df["total_cells"]

# =========================
# 5. quick sanity check
# =========================
print(prop_df.head())
print(prop_df.groupby(["Age_1", "anno_layer1"])["proportion"].mean().unstack())

# =========================
# 6. stats per cell type
# =========================
stats = []

for ct in celltypes_use:
    sub = prop_df[prop_df["anno_layer1"] == ct]

    g_adult = sub.loc[sub["Age_1"] == "adult", "proportion"]
    g_feotus = sub.loc[sub["Age_1"] == "feotus", "proportion"]

    _, pval = mannwhitneyu(g_adult, g_feotus, alternative="two-sided")

    stats.append({
        "celltype": ct,
        "adult_mean": g_adult.mean(),
        "feotus_mean": g_feotus.mean(),
        "pval": pval
    })

stats_df = pd.DataFrame(stats)
stats_df["padj"] = multipletests(stats_df["pval"], method="fdr_bh")[1]
stats_df["higher_in"] = stats_df.apply(
    lambda x: "adult" if x["adult_mean"] > x["feotus_mean"] else "feotus",
    axis=1
)

print(stats_df)

# =========================
# 7. plot
# =========================
plt.figure(figsize=(7, 4))

sns.boxplot(
    data=prop_df,
    x="anno_layer1",
    y="proportion",
    hue="Age_1",
    order=celltypes_use,
    showfliers=False
)

sns.stripplot(
    data=prop_df,
    x="anno_layer1",
    y="proportion",
    hue="Age_1",
    order=celltypes_use,
    dodge=True,
    color="black",
    size=5
)

handles, labels = plt.gca().get_legend_handles_labels()
plt.legend(handles[:2], labels[:2], title="Age_1", bbox_to_anchor=(1.02, 1), loc="upper left")

# add FDR
y_max = prop_df["proportion"].max()
for i, ct in enumerate(celltypes_use):
    padj = stats_df.loc[stats_df["celltype"] == ct, "padj"].iloc[0]
    plt.text(i, y_max * 1.05, f"FDR={padj:.2e}", ha="center", fontsize=9)

plt.ylim(0, y_max * 1.18)
plt.xlabel("")
plt.ylabel("Proportion")
plt.title("Per-sample cell-type proportions")
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
obs_df = adata_pb.obs[['anno_layer1', 'anatomical_site']]
frequency_df = obs_df.groupby(['anno_layer1', 'anatomical_site']).size().unstack(fill_value=0)
percentage_df = frequency_df.div(frequency_df.sum(axis=1), axis=0) * 100
percentage_df = percentage_df.iloc[::-1]

fig = plt.figure(figsize=(10, 3))
ax = percentage_df.plot(kind='barh', stacked=True, ax=fig.gca())  

plt.title("Proportion of clusters across age / clusters")
plt.xlabel("Percentage (%)")
plt.ylabel("sample")
plt.legend(title='Clusters', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(False)  
plt.xticks(rotation=0)  

plt.tight_layout()
plt.tight_layout()
#plt.savefig("samplefreq.pdf", dpi=600)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
obs_df = adata_pb.obs[['anno_layer1', 'anatomical_site']]
frequency_df = obs_df.groupby(['anno_layer1', 'anatomical_site']).size().unstack(fill_value=0)
percentage_df = frequency_df.div(frequency_df.sum(axis=1), axis=0) * 100
percentage_df = percentage_df.iloc[::-1]

fig = plt.figure(figsize=(10, 3))
ax = percentage_df.plot(kind='barh', stacked=True, ax=fig.gca())  

plt.title("Proportion of clusters across age / clusters")
plt.xlabel("Percentage (%)")
plt.ylabel("sample")
plt.legend(title='Clusters', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(False)  
plt.xticks(rotation=0)  

plt.tight_layout()
plt.tight_layout()
#plt.savefig("samplefreq.pdf", dpi=600)
plt.show()

In [ ]:
adata_layer = adata[adata.obs["layer"].isin(["epicardium", "myocardium"])].copy()

In [ ]:
del adata_pb

In [ ]:
order = ["Mac0", "Mac1", "Mac2", "Mac3", "Mac4", "Mac5", "Mac6"]

adata_layer.obs["anno_layer1"] = pd.Categorical(
    adata_layer.obs["anno_layer1"],
    categories=order,
    ordered=True
)


import matplotlib.pyplot as plt
obs_df = adata_layer.obs[['layer', 'anno_layer1']]
frequency_df = obs_df.groupby(['layer', 'anno_layer1']).size().unstack(fill_value=0)
percentage_df = frequency_df.div(frequency_df.sum(axis=1), axis=0) * 100
percentage_df = percentage_df.iloc[::-1]

fig = plt.figure(figsize=(10, 3))
ax = percentage_df.plot(kind='barh', stacked=True, ax=fig.gca())  

plt.title("Proportion of clusters across layer / clusters")
plt.xlabel("Percentage (%)")
plt.ylabel("sample")
plt.legend(title='Clusters', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(False)  
plt.xticks(rotation=0)  

plt.tight_layout()
plt.tight_layout()
#plt.savefig("samplefreq.pdf", dpi=600)
plt.show()

In [ ]:
adata_mi = adata[adata.obs["Disease"].isin(["healthy", "myocardial infarction"])].copy()

In [ ]:
order = ["Mac0", "Mac1", "Mac2", "Mac3", "Mac4", "Mac5", "Mac6"]

adata_mi.obs["anno_layer1"] = pd.Categorical(
    adata_mi.obs["anno_layer1"],
    categories=order,
    ordered=True
)


import matplotlib.pyplot as plt
obs_df = adata_mi.obs[['Disease', 'anno_layer1']]
frequency_df = obs_df.groupby(['Disease', 'anno_layer1']).size().unstack(fill_value=0)
percentage_df = frequency_df.div(frequency_df.sum(axis=1), axis=0) * 100
percentage_df = percentage_df.iloc[::-1]

fig = plt.figure(figsize=(10, 3))
ax = percentage_df.plot(kind='barh', stacked=True, ax=fig.gca())  

plt.title("Proportion of clusters across Disease / clusters")
plt.xlabel("Percentage (%)")
plt.ylabel("sample")
plt.legend(title='Clusters', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(False)  
plt.xticks(rotation=0)  

plt.tight_layout()
plt.tight_layout()
#plt.savefig("samplefreq.pdf", dpi=600)
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
import anndata as ad
X = adata_mi.layers["counts"] if "counts" in adata_mi.layers else adata_mi.X
X = X.tocsr() if sp.issparse(X) else sp.csr_matrix(X)

samples = adata_mi.obs["sample"].astype(str).to_numpy()
sample_order = pd.Series(samples).drop_duplicates().tolist()

pb_mat = np.vstack([
    np.asarray(X[samples == s].sum(axis=0)).ravel()
    for s in sample_order
])

obs_pb = (
    adata_mi.obs
    .drop_duplicates(subset=["sample"])
    .set_index("sample")
    .loc[sample_order, ["Disease"]]
    .copy()
)
obs_pb.index.name = None

adata_pb = ad.AnnData(
    X=pb_mat,
    obs=obs_pb,
    var=adata_mi.var.copy()
)
adata_pb.obs_names = sample_order

counts = pd.DataFrame(
    adata_pb.X,
    index=adata_pb.obs_names,
    columns=adata_pb.var_names
).T

counts.to_csv("pseudobulk_counts_healthy_vs_mi.tsv", sep="\t")
adata_pb.obs.to_csv("pseudobulk_metadata_healthy_vs_mi.tsv", sep="\t")

In [ ]:
adata.obs["anatomical_site"] = adata.obs["anatomical_site"].astype("category")

pal_map = {
    "aortic valve": "#020bf7",
    "apex": "#fcca5d",
    "base": "#00ff08",
    "infarct region": "#ff00f2",
    "interventricular septum": "#a200ff",
    "left atrium": "#ededab",
    "left ventricle": "#000000",
    "right atrium": "#ffff00",
    "right ventricle": "#90bd8c",
}

cats = adata.obs["anatomical_site"].cat.categories
sc.pl.umap(adata, color="anatomical_site", palette=[pal_map[c] for c in cats], legend_loc="right margin", size  = 30 )

In [ ]:
adata.obs["Disease"] = adata.obs["Disease"].astype("category")

pal_map =  {
    "healthy": "#009E73",
    "fibrosis": "#E69F00",
    "fibrocalcification": "#56B4E9",
    "heart failure": "#D55E00",
    "ischaemic cardiomyopathy": "#0072B2",
    "myocardial infarction": "#CC79A7",
    "non-ischaemic cardiomyopathy": "#999999",
}


cats = adata.obs["Disease"].cat.categories
sc.pl.umap(adata, color="Disease", palette=[pal_map[c] for c in cats], legend_loc="right margin", size  = 30 )

In [ ]:
#order = ["Mac1", "Mac0", "Mac2", "Mac6", "Mac3", "Mac4", "Mac5"]

adata.obs["anno_layer1"] = pd.Categorical(
    adata.obs["anno_layer1"],
    categories=order,
    ordered=True
)

sc.pl.matrixplot(
    adata,
    var_names=[
        'NRP1'
    ],
    groupby="anno_layer1",
    standard_scale="var",
    cmap="RdBu_r"
)

In [ ]:
bulk50 = [
    "CCL3.AS1", "CCL18", "LYVE1", "F13A1", "CCL13", "LINC02688", "P2RY12",
    "SPATA8", "ZSWIM4", "MCF2L2P1", "ITGA9", "CCL23", "RAMP1", "P2RY14",
    "CCL26", "MIR6503", "GLDN", "CLEC4GP1", "SAA1", "TMIGD3", "IGHE",
    "HRH1", "PDK4", "CCDC89", "SOCS1", "MIR5581", "RBP4", "ZNF436",
    "ADAMTS2", "DAAM2", "GGT5", "IL17RB", "MED28P3", "ANGPTL4", "FKBP5",
    "MS4A6E", "MRC1", "VSIG4", "VCAM1", "C2", "EHD3", "C1QB", "MAOA",
    "ARC", "MANSC1", "CD200R1", "SESN1.1", "TFPI", "ENHO", "SLCO4A1"
]

sc.tl.score_genes(adata, gene_list=bulk50, score_name="bulk50_score")

In [ ]:
mac4_top50 = [
    "SNX29", "PDGFC", "IQGAP2", "JUNB", "DPYD", "LINC00278", "HRH1",
    "KLF4", "FOSB", "AL162414.1", "PMP22", "FHIT", "ELMO1", "F13A1",
    "FKBP5", "DUSP1", "MS4A4E", "ZFP36", "C1QA", "IER2", "JUN", "MYO5A",
    "TCF12", "C1QB", "MTSS1", "ZBTB20", "TBC1D5", "EPS15", "FTX",
    "ARHGAP15", "MED13L", "PLXDC2", "CELF2", "HDAC9", "AC007952.4",
    "STARD13", "LRMDA", "EXOC4", "TRPS1", "AP2A2", "DDX5", "SGMS1",
    "WDFY2", "FOXP1", "HS3ST2", "FOXN3", "PPM1L", "CERS6", "ARID1B",
    "SLC9A9"
]

mac4_top50 = [g for g in mac4_top50 if g in adata.var_names]
sc.tl.score_genes(adata, gene_list=mac4_top50, score_name="mac4_top50_score")

In [ ]:
sc.pl.matrixplot(adata, var_names=["NRP1","bulk50_score", "mac4_top50_score"],
                groupby = 'anno_layer1', cmap = 'RdBu_r', standard_scale = 'var')

In [ ]:
adata

In [ ]:
sc.pl.umap(adata, 
          color = ['Disease', 'anno_layer1', 'anatomical_site', 'layer', 'project_id', 'Age_1' ], size = 30,
           color_map = 'RdBu_r', ncols = 1
          )

In [ ]:
import matplotlib.pyplot as plt
obs_df = adata.obs[['anatomical_site', 'anno_layer1']]
frequency_df = obs_df.groupby(['anatomical_site', 'anno_layer1']).size().unstack(fill_value=0)
percentage_df = frequency_df.div(frequency_df.sum(axis=1), axis=0) * 100
percentage_df = percentage_df.iloc[::-1]

fig = plt.figure(figsize=(10, 3))
ax = percentage_df.plot(kind='barh', stacked=True, ax=fig.gca())  

plt.title("Proportion of clusters across anatomical_site / leiden")
plt.xlabel("Percentage (%)")
plt.ylabel("sample")
plt.legend(title='Clusters', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(False)  
plt.xticks(rotation=0)  

plt.tight_layout()
plt.tight_layout()
#plt.savefig("samplefreq.pdf", dpi=600)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
obs_df = adata.obs[['Age_1', 'anno_layer1']]
frequency_df = obs_df.groupby(['Age_1', 'anno_layer1']).size().unstack(fill_value=0)
percentage_df = frequency_df.div(frequency_df.sum(axis=1), axis=0) * 100
percentage_df = percentage_df.iloc[::-1]

fig = plt.figure(figsize=(10, 3))
ax = percentage_df.plot(kind='barh', stacked=True, ax=fig.gca())  

plt.title("Proportion of clusters across Disease / anatomical sites")
plt.xlabel("Percentage (%)")
plt.ylabel("sample")
plt.legend(title='Clusters', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(False)  
plt.xticks(rotation=0)  

plt.tight_layout()
plt.tight_layout()
#plt.savefig("samplefreq.pdf", dpi=600)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
obs_df = adata.obs[['anatomical_site', 'Disease']]
frequency_df = obs_df.groupby(['anatomical_site', 'Disease']).size().unstack(fill_value=0)
percentage_df = frequency_df.div(frequency_df.sum(axis=1), axis=0) * 100
percentage_df = percentage_df.iloc[::-1]

fig = plt.figure(figsize=(10, 3))
ax = percentage_df.plot(kind='barh', stacked=True, ax=fig.gca())  

plt.title("Proportion of clusters across Disease / anatomical sites")
plt.xlabel("Percentage (%)")
plt.ylabel("sample")
plt.legend(title='Clusters', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(False)  
plt.xticks(rotation=0)  

plt.tight_layout()
plt.tight_layout()
#plt.savefig("samplefreq.pdf", dpi=600)
plt.show()

In [ ]:
cats = adata.obs["Disease"].cat.categories
sc.pl.umap(adata, color="anatomical_site", palette=[pal_map[c] for c in cats], legend_loc="right margin", size  = 30 )

In [ ]:
import matplotlib.pyplot as plt

# 1) define a disease palette (colorblind friendly-ish)
disease_colors = {
    "healthy": "#009E73",
    "fibrosis": "#E69F00",
    "fibrocalcification": "#56B4E9",
    "heart failure": "#D55E00",
    "ischaemic cardiomyopathy": "#0072B2",
    "myocardial infarction": "#CC79A7",
    "non-ischaemic cardiomyopathy": "#999999",
}

# 2) set category order (optional but recommended)
disease_order = [d for d in disease_colors if d in adata.obs["Disease"].unique()]
adata.obs["Disease"] = adata.obs["Disease"].astype("category")
adata.obs["Disease"] = adata.obs["Disease"].cat.reorder_categories(disease_order, ordered=True)

# --- YOUR PLOT (percent stacked barh) ---
obs_df = adata.obs[["anatomical_site", "Disease"]]
frequency_df = obs_df.groupby(["anatomical_site", "Disease"]).size().unstack(fill_value=0)

# keep columns in disease_order and build the palette list in the same order
frequency_df = frequency_df.reindex(columns=disease_order, fill_value=0)
pal_list = [disease_colors[d] for d in disease_order]

percentage_df = frequency_df.div(frequency_df.sum(axis=1), axis=0) * 100
percentage_df = percentage_df.iloc[::-1]

fig, ax = plt.subplots(figsize=(10, 3))
percentage_df.plot(kind="barh", stacked=True, ax=ax, color=pal_list)

ax.set_title("Proportion of clusters across Disease / anatomical sites")
ax.set_xlabel("Percentage (%)")
ax.set_ylabel("sample")
ax.legend(title="Disease", bbox_to_anchor=(1.05, 1), loc="upper left")
ax.grid(False)
ax.tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
adata1

In [ ]:
import decoupler as dc

In [ ]:
collectri = dc.op.collectri(organism='human')
collectri

In [ ]:
dc.mt.ulm(data=adata, net=collectri)

In [ ]:
score = dc.pp.get_obsm(adata=adata, key='score_ulm')
score

In [ ]:
df = dc.tl.rankby_group(
    adata=score,
    groupby='anno_layer1',
    reference='rest',
    method='t-test_overestim_var'
)
df = df[(df['stat'] > 0) & (df['padj'] < 0.01)]
df

In [ ]:
n_markers = 5
source_markers = (
    df
    .groupby('group')
    .head(n_markers)
    .drop_duplicates('name')
    .groupby('group')
    ['name']
    .apply(lambda x: list(x))
    .to_dict()
)

In [ ]:
#sc.tl.dendrogram(score,   groupby='leiden_res0_4')
sc.tl.dendrogram(score,   groupby='anno_layer1')

In [ ]:
sc.pl.dotplot(adata,
            var_names = ['LYVE1', 'NRP1', 'F13A1', 'FOLR2', 'CSTB'], groupby = 'anno_layer1')

In [ ]:
sc.pl.matrixplot(
    adata=score,
    var_names=source_markers,
    groupby='anno_layer1',
   dendrogram=True,
    standard_scale= 'var' ,
    colorbar_title='Z-scaled scores',
    cmap='RdBu_r',
    #save = 'dc_day5'
)